# Retrieval API Evaluation (File-Level)

This notebook runs the API-based retrieval evaluation script and displays:
- Hit@k
- Precision@k
- Recall@k
- MRR

It uses the existing endpoints and current ground-truth file.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
from pprint import pprint

In [2]:
# Adjust if your API uses another port
BASE_URL = "http://127.0.0.1:8001/api/v1"
K = 5
MODE = "unified_project"  # or "project_per_file"
KEEP_PROJECTS = False

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SCRIPT_PATH = ROOT / "scripts" / "evaluate_retrieval_api.py"
REPORT_PATH = ROOT / "results" / "retrieval_api_eval" / "report.json"

print("ROOT:", ROOT)
print("SCRIPT_PATH exists:", SCRIPT_PATH.exists())
print("REPORT_PATH:", REPORT_PATH)

ROOT: D:\Qbrainpython\rag_lab
SCRIPT_PATH exists: True
REPORT_PATH: D:\Qbrainpython\rag_lab\results\retrieval_api_eval\report.json


In [3]:
cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    "--base-url", BASE_URL,
    "--k", str(K),
    "--mode", MODE,
]
if KEEP_PROJECTS:
    cmd.append("--keep-projects")

print("Running:\n", " ".join(cmd))
result = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
print("\nSTDOUT:\n", result.stdout)
if result.stderr:
    print("\nSTDERR:\n", result.stderr)
print("\nExit code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError("Evaluation script failed. Check logs above.")

Running:
 C:\Users\ahmad\Miniconda3\python.exe D:\Qbrainpython\rag_lab\scripts\evaluate_retrieval_api.py --base-url http://127.0.0.1:8001/api/v1 --k 5 --mode unified_project



STDOUT:
 [info] created unified project: 6d0b7e05-d2c3-4c67-9936-90e48dede73e
[info] uploading 2007 - ertms.pdf ...
[info] uploading 2008 - keepass.pdf ...
[info] uploading 2009 - inventory 2.0.pdf ...
[info] uploading 2010 - gparted.pdf ...
[info] uploading JDECo_SRS.docx[1].pdf ...

=== Retrieval API Evaluation ===
mode           : unified_project
queries        : 60
hit@5       : 1.0000
precision@5 : 0.2000
recall@5    : 1.0000
mrr            : 1.0000
report         : results\retrieval_api_eval\report.json


Exit code: 0


In [4]:
if not REPORT_PATH.exists():
    raise FileNotFoundError(f"Report not found: {REPORT_PATH}")

report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
summary = report.get("summary", {})

print("=== Summary ===")
pprint(summary)

=== Summary ===
{'hit@k': 1.0, 'mrr': 1.0, 'precision@k': 0.2, 'queries': 60, 'recall@k': 1.0}


In [5]:
per_query = report.get("per_query", [])
print("Queries:", len(per_query))
print("\nFirst 10 query results:")
for row in per_query[:10]:
    print({
        "question_id": row.get("question_id"),
        "relevant_file": row.get("relevant_file"),
        "ranked_top5": row.get("ranked_files", [])[:5],
        "hit@k": row.get("hit@k"),
        "rr": row.get("rr"),
    })

Queries: 60

First 10 query results:
{'question_id': 'ERTMS_1', 'relevant_file': '6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae971244f98901bc44bf074b54e.pdf', 'ranked_top5': ['6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae971244f98901bc44bf074b54e.pdf'], 'hit@k': 1.0, 'rr': 1.0}
{'question_id': 'ERTMS_2', 'relevant_file': '6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae971244f98901bc44bf074b54e.pdf', 'ranked_top5': ['6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae971244f98901bc44bf074b54e.pdf'], 'hit@k': 1.0, 'rr': 1.0}
{'question_id': 'ERTMS_3', 'relevant_file': '6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae971244f98901bc44bf074b54e.pdf', 'ranked_top5': ['6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae971244f98901bc44bf074b54e.pdf'], 'hit@k': 1.0, 'rr': 1.0}
{'question_id': 'ERTMS_4', 'relevant_file': '6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae971244f98901bc44bf074b54e.pdf', 'ranked_top5': ['6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae971244f98901bc44bf074b54e.pdf'], 'hit@k': 1.0, 'rr': 1.0}
{'q

In [6]:
# Optional detailed table if pandas is available
try:
    import pandas as pd
    df = pd.DataFrame(per_query)
    display(df.head(20))
except Exception as e:
    print("pandas not available or failed to render table:", e)

,question_id,question,relevant_file,ranked_files,hit@k,precision@k,recall@k,rr
0,ERTMS_1,"According to the ERTMS document, what does thi...",6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae97...,[6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae9...,1.0,0.2,1.0,1.0
1,ERTMS_2,According to the ERTMS document requirements n...,6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae97...,[6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae9...,1.0,0.2,1.0,1.0
2,ERTMS_3,According to the ERTMS document requirements n...,6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae97...,[6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae9...,1.0,0.2,1.0,1.0
3,ERTMS_4,"According to the ERTMS requirements, what info...",6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae97...,[6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae9...,1.0,0.2,1.0,1.0
4,ERTMS_5,"According to the ERTMS requirements, which mov...",6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae97...,[6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae9...,1.0,0.2,1.0,1.0
5,ERTMS_6,"According to the ERTMS requirements, what is t...",6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae97...,[6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae9...,1.0,0.2,1.0,1.0
6,ERTMS_7,According to the ETCS application levels in th...,6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae97...,[6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae9...,1.0,0.2,1.0,1.0
7,ERTMS_8,According to the ETCS application levels in th...,6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae97...,[6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae9...,1.0,0.2,1.0,1.0
8,ERTMS_9,"According to the ERTMS level-transition rule, ...",6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae97...,[6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae9...,1.0,0.2,1.0,1.0
9,ERTMS_10,"According to the ERTMS requirements, with whic...",6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae97...,[6d0b7e05-d2c3-4c67-9936-90e48dede73e_f97dbae9...,1.0,0.2,1.0,1.0
